# MapBiomas Fire — Validação de Export GCS (Peru)

Verifica se os **chunks (M1)** e **COGs (M2)** do monitor Sentinel-2 do Peru foram exportados corretamente para o GCS.

## Estrutura validada

| Campo | Valor |
|-------|-------|
| Bucket | `mapbiomas-fire` |
| Raiz | `sudamerica/peru/CATALOG_01/LIBRARY_IMAGES/SENTINEL2/MONTHLY/{MOSAIC}/{YYYY_MM}/` |
| Métodos | `MINNBR`, `MINNBR_BUFFER` |
| Bandas | `blue`, `green`, `red`, `nir`, `swir1`, `swir2`, `dayOfYear` |
| Sensores | `SENTINEL2` |

**Zero download**: só metadados (contagem, tamanhos).

In [1]:
# @title 1. Setup — Ambiente e Autenticação
import os, sys, json, datetime, time, re, textwrap, warnings
warnings.filterwarnings('ignore')
from collections import defaultdict

IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ
print(f'Ambiente: {"Google Colab" if IS_COLAB else "Local"}')

if IS_COLAB:
    !pip install -q gcsfs pandas tabulate 2>&1 | tail -1
    from google.colab import auth
    auth.authenticate_user()
    print('GCP autenticado via Colab.')
else:
    print('Assumindo Application Default Credentials (ADC) local.')

import gcsfs
import pandas as pd

fs = gcsfs.GCSFileSystem()
print('gcsfs pronto.')

Ambiente: Google Colab
GCP autenticado via Colab.
gcsfs pronto.


In [16]:
# @title 2. Configuração
BUCKET = 'mapbiomas-fire'
COUNTRY = 'bolivia'
SENSOR = 'LANDSAT' # ['SENTINEL2', 'LANDSAT']
PERIOD = 'ANNUALLY' # ['ANNUALLY', 'MONTHLY']
MOSAIC_METHODS = ['MINNBR'] # ['MINNBR', 'MINNBR_BUFFER']

# Caminho base atualizado
GCS_BASE = f'{BUCKET}/sudamerica/{COUNTRY}/CATALOG_01/LIBRARY_IMAGES/{SENSOR}/{PERIOD}'
BANDS = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'dayOfYear']

# Gera o nome base do mosaico
def mosaic_name(year, month_or_none, band, mosaic):
    m = mosaic.lower()
    if PERIOD == 'ANNUALLY':
        return f'image_{COUNTRY}_fire_{SENSOR.lower()}_{m}_{band}_{year}'
    else:
        return f'image_{COUNTRY}_fire_{SENSOR.lower()}_{m}_{band}_{year}_{month_or_none:02d}'

print(f'Bucket: {BUCKET}')
print(f'Base: {GCS_BASE}')
print(f'Métodos: {MOSAIC_METHODS}')
print(f'Bandas: {BANDS}')

Bucket: mapbiomas-fire
Base: mapbiomas-fire/sudamerica/bolivia/CATALOG_01/LIBRARY_IMAGES/LANDSAT/ANNUALLY
Métodos: ['MINNBR']
Bandas: ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'dayOfYear']


In [19]:
# @title 3. Varredura do GCS
def scan_gcs(mosaic_methods=None, bands=None):
    """Escaneia GCS e retorna lista de dicts com metadados de chunks e COGs."""
    mosaic_methods = mosaic_methods or MOSAIC_METHODS
    bands = bands or BANDS
    rows = []
    total_periods = 0

    for mosaic in mosaic_methods:
        base = f'{GCS_BASE}/{mosaic}'

        # Lista diretórios de tempo (YYYY_MM para mensal ou YYYY para anual)
        try:
            entries = fs.ls(base)
            pattern = r'^\d{4}$' if PERIOD == 'ANNUALLY' else r'^\d{4}_\d{2}$'
            periods = sorted([
                e.rstrip('/').split('/')[-1]
                for e in entries
                if fs.isdir(e) and re.match(pattern, e.rstrip('/').split('/')[-1])
            ])
        except Exception as e:
            print(f'  [SKIP] {mosaic}: não foi possível listar — {e}')
            continue

        total_periods += len(periods)
        print(f'  [{mosaic}] {len(periods)} pastas de tempo encontradas ({PERIOD})')

        for p in periods:
            # --- CHUNKS ---
            chunk_prefix = f'{base}/{p}/CHUNKS'
            chunk_files = []
            try:
                all_files = fs.find(chunk_prefix)
                chunk_files = [f for f in all_files if f.endswith('.tif')]
            except FileNotFoundError:
                pass  # Pasta CHUNKS não existe
            except Exception as e:
                print(f'    [WARN] {p} CHUNKS: {e}')

            # --- COGs ---
            cog_prefix = f'{base}/{p}/COG'
            cog_files = []
            try:
                all_cogs = fs.find(cog_prefix)
                cog_files = [f for f in all_cogs if f.endswith('_cog.tif') or f.endswith('.tif')]
            except FileNotFoundError:
                pass  # Pasta COG não existe
            except Exception as e:
                print(f'    [WARN] {p} COG: {e}')

            # Agrupa por banda (case-insensitive)
            for band in bands:
                b_lower = band.lower()

                # Filtra chunks desta banda
                band_chunks = [
                    f for f in chunk_files
                    if f'_{b_lower}_' in f.lower() or f'_{b_lower}.' in f.lower()
                ]

                # Filtra COGs desta banda
                band_cogs = [
                    f for f in cog_files
                    if f'_{b_lower}_' in f.lower() or f'_{b_lower}.' in f.lower() or f'_{b_lower}_cog' in f.lower()
                ]

                # Tamanhos
                sizes = []
                for f in band_chunks:
                    try:
                        sizes.append(fs.info(f)['size'])
                    except:
                        pass

                cog_size = 0
                if band_cogs:
                    try:
                        cog_size = fs.info(band_cogs[0])['size']
                    except:
                        pass

                rows.append({
                    'mosaic': mosaic,
                    'period_folder': p,
                    'year': int(p[:4]),
                    'month': int(p[5:7]) if PERIOD == 'MONTHLY' else None,
                    'band': band,
                    'n_chunks': len(band_chunks),
                    'chunks_bytes': sum(sizes),
                    'chunks_min': min(sizes) if sizes else 0,
                    'chunks_max': max(sizes) if sizes else 0,
                    'n_cogs': len(band_cogs),
                    'has_cog': len(band_cogs) > 0,
                    'cog_bytes': cog_size,
                })

    return rows, total_periods

print('Iniciando varredura do GCS...')
t0 = time.time()
records, n_months = scan_gcs()
elapsed = time.time() - t0
print(f'\nVarredura concluída em {elapsed:.1f}s')
print(f'Total: {len(records)} registros (bandas × períodos), {n_months} períodos/anos processados')

Iniciando varredura do GCS...
  [MINNBR] 1 pastas de tempo encontradas (ANNUALLY)

Varredura concluída em 0.2s
Total: 7 registros (bandas × períodos), 1 períodos/anos processados


In [20]:
# @title 4. Validação e Tabela Resumo
df = pd.DataFrame(records)

if df.empty:
    print('Nenhum dado encontrado no GCS.')
else:
    # Converte bytes → MB
    df['chunks_mb'] = (df['chunks_bytes'] / 1e6).round(1)
    df['chunks_avg_mb'] = (df['chunks_bytes'] / df['n_chunks'].replace(0, 1) / 1e6).round(2)
    df['cog_mb'] = (df['cog_bytes'] / 1e6).round(1)

    # Regras de validação
    def classify(row):
        if row['n_chunks'] == 0:
            return 'MISSING'
        if row['has_cog']:
            return 'OK'
        return 'CHUNKS_ONLY'

    df['status'] = df.apply(classify, axis=1)

    # Paleta de cores
    STATUS_COLORS = {
        'OK': 'background-color: #d4edda; color: #155724',
        'CHUNKS_ONLY': 'background-color: #fff3cd; color: #856404',
        'MISSING': 'background-color: #f8d7da; color: #721c24',
    }

    # --- TABELA DETALHADA ---
    display_cols = ['mosaic', 'period_folder', 'band', 'n_chunks', 'chunks_mb',
                    'chunks_avg_mb', 'has_cog', 'cog_mb', 'status']
    detail = df[display_cols].copy()

    styled = detail.style \
        .applymap(lambda s: STATUS_COLORS.get(s, ''), subset=['status']) \
        .format({
            'chunks_mb': '{:.1f}',
            'chunks_avg_mb': '{:.2f}',
            'cog_mb': '{:.1f}',
        }) \
        .set_caption(f'Validação Chunks e COGs — {COUNTRY} {SENSOR}')

    display(styled)

    # --- RESUMO POR PERÍODO ---
    print('\n' + '='*80)
    print('RESUMO POR PERÍODO')
    print('='*80)

    for mosaic in df['mosaic'].unique():
        sub = df[df['mosaic'] == mosaic]
        print(f'\n--- {mosaic} ---')
        for p in sorted(sub['period_folder'].unique(), reverse=True):
            row = sub[sub['period_folder'] == p]
            n_ok = (row['status'] == 'OK').sum()
            n_chunks_only = (row['status'] == 'CHUNKS_ONLY').sum()
            n_miss = (row['status'] == 'MISSING').sum()
            n_bands_total = len(row)

            if n_ok == n_bands_total:
                icon = '✅'
            elif n_chunks_only > 0:
                icon = '⚠️'
            else:
                icon = '❌'

            chunks_only_str = f' | {n_chunks_only} só chunks' if n_chunks_only else ''
            missing_str = f' | {n_miss} faltando' if n_miss else ''
            print(f'  {icon} {p}: {n_ok}/{n_bands_total} COGs OK{chunks_only_str}{missing_str}')

,mosaic,period_folder,band,n_chunks,chunks_mb,chunks_avg_mb,has_cog,cog_mb,status
0,MINNBR,1995,blue,9,5549.3,616.59,False,0.0,CHUNKS_ONLY
1,MINNBR,1995,green,9,5782.0,642.44,False,0.0,CHUNKS_ONLY
2,MINNBR,1995,red,9,5992.5,665.83,False,0.0,CHUNKS_ONLY
3,MINNBR,1995,nir,9,7832.8,870.31,False,0.0,CHUNKS_ONLY
4,MINNBR,1995,swir1,9,8037.9,893.10,False,0.0,CHUNKS_ONLY
5,MINNBR,1995,swir2,9,7056.2,784.02,False,0.0,CHUNKS_ONLY
6,MINNBR,1995,dayOfYear,12,4843.0,403.58,False,0.0,CHUNKS_ONLY



RESUMO POR PERÍODO

--- MINNBR ---
  ⚠️ 1995: 0/7 COGs OK | 7 só chunks


In [23]:
# @title 5. Anomalias
print('='*80)
print('ANOMALIAS DETECTADAS')
print('='*80)

anomalies = []

# --- 1. Chunks sem COG ---
chunks_only = df[df['status'] == 'CHUNKS_ONLY']
for _, row in chunks_only.iterrows():
    msg = f'  ⚠️  [{row["mosaic"]}] {row["period_folder"]} / {row["band"]}: {row["n_chunks"]} chunks, mas SEM COG'
    print(msg)
    anomalies.append(msg)

# --- 2. Bandas completamente faltando ---
missing = df[df['status'] == 'MISSING']
for _, row in missing.iterrows():
    msg = f'  ❌ [{row["mosaic"]}] {row["period_folder"]} / {row["band"]}: 0 chunks (banda ausente)'
    print(msg)
    anomalies.append(msg)

# --- 3. Chunks com tamanho anômalo (fora de 3× std de tempos vizinhos) ---
for mosaic in df['mosaic'].unique():
    for band in BANDS:
        sub = df[(df['mosaic'] == mosaic) & (df['band'] == band)]
        sizes = sub[sub['n_chunks'] > 0]['chunks_avg_mb']
        if len(sizes) >= 3:
            mean = sizes.mean()
            std = sizes.std()
            outliers = sub[(sub['n_chunks'] > 0) &
                          ((sub['chunks_avg_mb'] < mean - 3*std) |
                           (sub['chunks_avg_mb'] > mean + 3*std))]
            for _, row in outliers.iterrows():
                msg = (f'  🔍 [{row["mosaic"]}] {row["period_folder"]} / {row["band"]}: '
                       f'tamanho médio {row["chunks_avg_mb"]}MB '
                       f'(média {mean:.2f}MB, std {std:.2f}MB)')
                print(msg)
                anomalies.append(msg)

if not anomalies:
    print('  Nenhuma anomalia detectada. ✅')
else:
    print(f'\n  Total: {len(anomalies)} anomalia(s)')

ANOMALIAS DETECTADAS
  ⚠️  [MINNBR] 1995 / blue: 9 chunks, mas SEM COG
  ⚠️  [MINNBR] 1995 / green: 9 chunks, mas SEM COG
  ⚠️  [MINNBR] 1995 / red: 9 chunks, mas SEM COG
  ⚠️  [MINNBR] 1995 / nir: 9 chunks, mas SEM COG
  ⚠️  [MINNBR] 1995 / swir1: 9 chunks, mas SEM COG
  ⚠️  [MINNBR] 1995 / swir2: 9 chunks, mas SEM COG
  ⚠️  [MINNBR] 1995 / dayOfYear: 12 chunks, mas SEM COG

  Total: 7 anomalia(s)


In [24]:
# @title 5b. Completude por Ano
print("=" * 80)
print("COMPLETUDE POR ANO")
print("=" * 80)

today = datetime.date.today()
current_year = today.year
current_month = today.month

# Identifica os períodos com dados
found = df[["mosaic", "year", "period_folder"]].drop_duplicates()

year_rows = []
for mosaic in sorted(found["mosaic"].unique()):
    sub = found[found["mosaic"] == mosaic]
    years = sorted(sub["year"].unique())

    for year in years:
        is_current = (year == current_year)

        if PERIOD == 'ANNUALLY':
            # Para o padrão anual, cada ano espera apenas 1 registro
            n_expected = 1
            existing = set(sub[sub["year"] == year]["period_folder"])
            expected_periods = [f"{year}"]
        else:
            # Para o padrão mensal
            if year < current_year:
                n_expected = 12
            elif is_current:
                n_expected = max(0, current_month - 1)
            else:
                n_expected = 0
            existing = set(sub[sub["year"] == year]["period_folder"])
            expected_periods = [f"{year}_{m:02d}" for m in range(1, n_expected + 1)]

        found_list = [p for p in expected_periods if p in existing]
        missing_list = [p for p in expected_periods if p not in existing]

        if n_expected == 0:
            pct = 100.0
        else:
            pct = len(found_list) / n_expected * 100

        if n_expected == 0:
            icon = chr(0x2795)
            status = "FUTURE"
        elif len(found_list) >= n_expected:
            icon = chr(0x2705)
            status = "COMPLETE"
        elif len(found_list) >= n_expected / 2:
            icon = chr(0x26A0) + chr(0xFE0F)
            status = "PARTIAL"
        else:
            icon = chr(0x274C)
            status = "SPARSE"

        unit_name = "anos" if PERIOD == 'ANNUALLY' else "meses"
        extra = " (Ano corrente)" if is_current else ""
        print(f"  {icon} [{mosaic}] {year}: {len(found_list)}/{n_expected} {unit_name} ({pct:.0f}%){extra}")
        if missing_list:
            print(f"       Faltando: {', '.join(missing_list)}")

        year_rows.append({
            'mosaic': mosaic,
            'year': year,
            'year_current': is_current,
            'periods_found': len(found_list),
            'periods_expected': n_expected,
            'completude_pct': f'{round(pct)}%',
            'status': status,
            'missing_periods': ', '.join(missing_list) if missing_list else '-',
        })

df_year = pd.DataFrame(year_rows)
print(f"\nTotal: {len(df_year)} anos x mosaicos analisados")

COMPLETUDE POR ANO
  ✅ [MINNBR] 1995: 1/1 anos (100%)

Total: 1 anos x mosaicos analisados


In [ ]:
# @title 6. Download do Relatório
import base64, textwrap
from IPython.display import display, HTML

if not df.empty:
    timestamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')

    # --- CSV ---
    csv_bytes = df.to_csv(index=False).encode('utf-8')
    csv_b64 = base64.b64encode(csv_bytes).decode()
    csv_name = f'validation_report_{timestamp}.csv'

    # --- HTML colorido ---
    _C = {'OK': '#d4edda', 'CHUNKS_ONLY': '#fff3cd', 'MISSING': '#f8d7da'}
    html_table = df.style \
        .applymap(lambda s: f'background-color: {_C.get(s, "")}', subset=['status']) \
        .format({'chunks_bytes': '{:,}', 'chunks_min': '{:,}',
                 'chunks_max': '{:,}', 'cog_bytes': '{:,}'}) \
        .hide(axis='index').to_html()

    # --- Tabela de completude por ano ---
    if not df_year.empty:
        _CY = {'COMPLETE': '#d4edda', 'PARTIAL': '#fff3cd', 'SPARSE': '#f8d7da', 'FUTURE': '#e2e3e5'}
        html_year = df_year.style \
            .applymap(lambda s: f'background-color: {_CY.get(s, "")}', subset=['status']) \
            .hide(axis='index').to_html()
    else:
        html_year = '<p>Sem dados de ano.</p>'


    html_content = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>Validação GCS — Peru Fire {timestamp}</title>
<style>
body {{ font-family: sans-serif; margin: 20px; }}
table {{ border-collapse: collapse; }}
td, th {{ padding: 4px 8px; border: 1px solid #ccc; font-size: 12px; }}
th {{ background: #f0f0f0; }}
</style></head><body>
<h2>Validação Export GCS — Peru Fire Monitor</h2>
<p>Gerado em: {timestamp}</p>
{html_table}
{html_year}
</body></html>'''
    html_b64 = base64.b64encode(html_content.encode()).decode()
    html_name = f'validation_report_{timestamp}.html'

    # Exibe a tabela colorida inline
    display(HTML(f'<h3>Relatório de Validação — {timestamp}</h3>' + html_table))
    if not df_year.empty:
        display(HTML(f'<h4>Completude por Ano</h4>' + html_year))

    # Botões de download
    display(HTML(f'''
    <div style="margin: 20px 0;">
      <a download="{csv_name}" href="data:text/csv;base64,{csv_b64}"
         style="padding: 10px 20px; background: #28a745; color: white; text-decoration: none; border-radius: 4px; margin-right: 10px;">
        📥 Download CSV
      </a>
      <a download="{html_name}" href="data:text/html;base64,{html_b64}"
         style="padding: 10px 20px; background: #007bff; color: white; text-decoration: none; border-radius: 4px;">
        📥 Download HTML
      </a>
    </div>
    '''))
else:
    print('Nada a exportar (DataFrame vazio).')

---
**Fim da validação.**

Se encontrou anomalias, revise os meses/métodos sinalizados e reexecute M1 (export) ou M2 (mosaic) conforme necessário.